In [3]:
from pathlib import Path
import duckdb

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

database_path = (
    project_root / "data" / "warehouse" / "customer_loyalty.duckdb"
)

if not database_path.is_file():
    raise FileNotFoundError(f"Database not found: {database_path}")

con = duckdb.connect(str(database_path))

print(f"Connected to: {database_path}")

Connected to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/warehouse/customer_loyalty.duckdb


In [4]:
con.sql("""
    SELECT
        'transactions' AS source,
        MIN(DAY) AS first_recorded_day,
        MAX(DAY) AS last_recorded_day
    FROM transactions

    UNION ALL

    SELECT
        'coupon_redemptions' AS source,
        MIN(DAY) AS first_recorded_day,
        MAX(DAY) AS last_recorded_day
    FROM coupon_redemptions
""").df()

,source,first_recorded_day,last_recorded_day
0,transactions,1,711
1,coupon_redemptions,225,704


In [5]:
con.sql("""
    SELECT
        CAMPAIGN,
        DESCRIPTION,
        START_DAY,
        END_DAY
    FROM campaigns
    WHERE END_DAY > (
        SELECT MAX(DAY)
        FROM coupon_redemptions
    )
    ORDER BY END_DAY, CAMPAIGN
""").df()

,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY
0,15,TypeC,547,708
1,24,TypeB,659,719


In [6]:
# Conservative baseline cutoff—not a confirmed data-collection end date.
outcome_cutoff_day = 704

eligible_campaigns = con.sql(
    """
    SELECT CAMPAIGN, DESCRIPTION, START_DAY, END_DAY
    FROM campaigns
    WHERE END_DAY <= ?
    ORDER BY START_DAY, CAMPAIGN
    """,
    params=[outcome_cutoff_day]
).df()

print(f"Eligible campaigns: {len(eligible_campaigns)}")

Eligible campaigns: 28


In [7]:
eligible_household_campaigns = con.sql(
    """
    SELECT
        household_key,
        CAMPAIGN,
        DESCRIPTION,
        START_DAY,
        END_DAY
    FROM household_campaigns
    WHERE END_DAY <= ?
    ORDER BY START_DAY, CAMPAIGN, household_key
    """,
    params=[outcome_cutoff_day]
).df()

print(f"Eligible household–campaign pairs: {len(eligible_household_campaigns):,}")
print(
    f"Distinct households: "
    f"{eligible_household_campaigns['household_key'].nunique():,}"
)

Eligible household–campaign pairs: 7,091
Distinct households: 1,578


In [8]:
con.register("eligible_pairs", eligible_household_campaigns)

campaign_labels = con.sql("""
    SELECT
        e.*,
        CASE WHEN EXISTS (
            SELECT 1
            FROM coupon_redemptions AS r
            WHERE r.household_key = e.household_key
              AND r.CAMPAIGN = e.CAMPAIGN
              AND r.DAY BETWEEN e.START_DAY AND e.END_DAY
        )
        THEN 1 ELSE 0 END AS redeemed
    FROM eligible_pairs AS e
""").df()

print(f"Labeled examples: {len(campaign_labels):,}")
print(campaign_labels["redeemed"].value_counts().sort_index())

Labeled examples: 7,091
redeemed
0    6211
1     880
Name: count, dtype: int64


In [9]:
lookback_days = 90

campaign_labels["history_start_day"] = (
    campaign_labels["START_DAY"] - lookback_days
)

campaign_labels["history_end_day"] = (
    campaign_labels["START_DAY"] - 1
)

campaign_labels[
    [
        "household_key",
        "CAMPAIGN",
        "START_DAY",
        "history_start_day",
        "history_end_day"
    ]
].head()

,household_key,CAMPAIGN,START_DAY,history_start_day,history_end_day
0,17,26,224,134,223
1,22,26,224,134,223
2,27,26,224,134,223
3,32,26,224,134,223
4,39,26,224,134,223


In [11]:
transaction_bounds = con.sql("""
    SELECT MIN(DAY), MAX(DAY)
    FROM transactions
""").fetchall()[0]

first_transaction_day, last_transaction_day = transaction_bounds

outside_history_coverage = (
    (campaign_labels["history_start_day"] < first_transaction_day)
    | (campaign_labels["history_end_day"] > last_transaction_day)
)

print(
    f"Examples with history windows outside transaction coverage: "
    f"{outside_history_coverage.sum():,}"
)

Examples with history windows outside transaction coverage: 0


In [12]:
con.register("modeling_pairs", campaign_labels)

basket_features = con.sql("""
    SELECT
        m.household_key,
        m.CAMPAIGN,
        COUNT(DISTINCT t.BASKET_ID) AS baskets_90d
    FROM modeling_pairs AS m
    LEFT JOIN transactions AS t
        ON m.household_key = t.household_key
       AND t.DAY BETWEEN m.history_start_day AND m.history_end_day
    GROUP BY m.household_key, m.CAMPAIGN
    ORDER BY m.CAMPAIGN, m.household_key
""").df()

print(f"Feature rows: {len(basket_features):,}")
basket_features.head()

Feature rows: 7,091


,household_key,CAMPAIGN,baskets_90d
0,68,1,66
1,93,1,20
2,105,1,21
3,293,1,14
4,529,1,2


In [13]:
modeling_table = campaign_labels.merge(
    basket_features,
    on=["household_key", "CAMPAIGN"],
    how="left",
    validate="one_to_one"
)

print(f"Modeling rows: {len(modeling_table):,}")
print(f"Missing basket counts: {modeling_table['baskets_90d'].isna().sum():,}")

Modeling rows: 7,091
Missing basket counts: 0


In [14]:
sales_features = con.sql("""
    SELECT
        m.household_key,
        m.CAMPAIGN,
        COALESCE(SUM(t.SALES_VALUE), 0) AS retailer_sales_90d
    FROM modeling_pairs AS m
    LEFT JOIN transactions AS t
        ON m.household_key = t.household_key
       AND t.DAY BETWEEN m.history_start_day AND m.history_end_day
    GROUP BY m.household_key, m.CAMPAIGN
    ORDER BY m.CAMPAIGN, m.household_key
""").df()

print(f"Feature rows: {len(sales_features):,}")
sales_features.head().round(2)

Feature rows: 7,091


,household_key,CAMPAIGN,retailer_sales_90d
0,68,1,845.49
1,93,1,687.12
2,105,1,802.52
3,293,1,745.79
4,529,1,109.52


In [15]:
modeling_table = modeling_table.merge(
    sales_features,
    on=["household_key", "CAMPAIGN"],
    how="left",
    validate="one_to_one"
)

print(f"Modeling rows: {len(modeling_table):,}")
print(
    f"Missing sales values: "
    f"{modeling_table['retailer_sales_90d'].isna().sum():,}"
)

Modeling rows: 7,091
Missing sales values: 0


In [16]:
recency_features = con.sql("""
    SELECT
        m.household_key,
        m.CAMPAIGN,
        m.START_DAY - MAX(t.DAY) AS recency_days_90d
    FROM modeling_pairs AS m
    LEFT JOIN transactions AS t
        ON m.household_key = t.household_key
       AND t.DAY BETWEEN m.history_start_day AND m.history_end_day
    GROUP BY m.household_key, m.CAMPAIGN, m.START_DAY
    ORDER BY m.CAMPAIGN, m.household_key
""").df()

print(f"Feature rows: {len(recency_features):,}")
recency_features.head()

Feature rows: 7,091


,household_key,CAMPAIGN,recency_days_90d
0,68,1,6
1,93,1,2
2,105,1,18
3,293,1,12
4,529,1,21


In [17]:
modeling_table = modeling_table.merge(
    recency_features,
    on=["household_key", "CAMPAIGN"],
    how="left",
    validate="one_to_one"
)

print(f"Modeling rows: {len(modeling_table):,}")
print(
    f"Examples with no transaction in the 90-day window: "
    f"{modeling_table['recency_days_90d'].isna().sum():,}"
)

Modeling rows: 7,091
Examples with no transaction in the 90-day window: 27


In [18]:
modeling_table["has_transactions_90d"] = (
    modeling_table["baskets_90d"] > 0
).astype(int)

history_mismatches = (
    modeling_table["recency_days_90d"].isna()
    != modeling_table["has_transactions_90d"].eq(0)
).sum()

print(
    f"Examples without transaction history: "
    f"{modeling_table['has_transactions_90d'].eq(0).sum():,}"
)
print(f"Basket/recency inconsistencies: {history_mismatches:,}")

Examples without transaction history: 27
Basket/recency inconsistencies: 0


In [19]:
modeling_table["avg_retailer_sales_per_basket_90d"] = (
    modeling_table["retailer_sales_90d"]
    / modeling_table["baskets_90d"].where(
        modeling_table["baskets_90d"] > 0
    )
)

modeling_table[
    [
        "household_key",
        "CAMPAIGN",
        "baskets_90d",
        "retailer_sales_90d",
        "avg_retailer_sales_per_basket_90d"
    ]
].head().round(2)

,household_key,CAMPAIGN,baskets_90d,retailer_sales_90d,avg_retailer_sales_per_basket_90d
0,17,26,23,823.07,35.79
1,22,26,13,770.11,59.24
2,27,26,16,120.21,7.51
3,32,26,20,527.19,26.36
4,39,26,11,804.74,73.16


In [20]:
numeric_feature_columns = [
    "baskets_90d",
    "retailer_sales_90d",
    "recency_days_90d",
    "has_transactions_90d",
    "avg_retailer_sales_per_basket_90d"
]

modeling_table[numeric_feature_columns].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
baskets_90d,7091.0,27.237766,23.44413,0.0,13.0,21.0,34.0,257.0
retailer_sales_90d,7091.0,861.271337,635.285926,0.0,396.605,712.18,1155.995,4104.61
recency_days_90d,7064.0,6.106172,8.934978,1.0,2.0,3.0,7.0,89.0
has_transactions_90d,7091.0,0.996192,0.061593,0.0,1.0,1.0,1.0,1.0
avg_retailer_sales_per_basket_90d,7064.0,37.387333,24.790842,1.88,20.701427,31.594281,47.251544,322.755


In [21]:
modeling_table.nlargest(5, "baskets_90d")[
    [
        "household_key",
        "CAMPAIGN",
        "history_start_day",
        "history_end_day",
        "baskets_90d",
        "retailer_sales_90d",
        "avg_retailer_sales_per_basket_90d"
    ]
].round(2)

,household_key,CAMPAIGN,history_start_day,history_end_day,baskets_90d,retailer_sales_90d,avg_retailer_sales_per_basket_90d
4189,2337,13,414,503,257,2255.55,8.78
1408,2337,7,308,397,246,2114.46,8.60
2428,2337,8,322,411,232,2087.28,9.00
3816,1467,13,414,503,221,592.79,2.68
4462,2337,14,441,530,215,1962.22,9.13


In [22]:
con.sql("""
    SELECT
        DAY,
        COUNT(DISTINCT BASKET_ID) AS baskets,
        COUNT(DISTINCT STORE_ID) AS stores,
        SUM(SALES_VALUE) AS retailer_sales
    FROM transactions
    WHERE household_key = 2337
      AND DAY BETWEEN 414 AND 503
    GROUP BY DAY
    ORDER BY baskets DESC, DAY
    LIMIT 10
""").df().round(2)

,DAY,baskets,stores,retailer_sales
0,432,9,1,92.58
1,423,8,1,48.59
2,437,8,1,38.95
3,501,8,1,95.88
4,418,7,1,38.87
5,446,7,1,31.74
6,498,7,1,43.83
7,441,6,1,112.04
8,447,6,1,36.09
9,499,6,1,42.59


In [23]:
active_day_features = con.sql("""
    SELECT
        m.household_key,
        m.CAMPAIGN,
        COUNT(DISTINCT t.DAY) AS active_days_90d
    FROM modeling_pairs AS m
    LEFT JOIN transactions AS t
        ON m.household_key = t.household_key
       AND t.DAY BETWEEN m.history_start_day AND m.history_end_day
    GROUP BY m.household_key, m.CAMPAIGN
""").df()

modeling_table = modeling_table.merge(
    active_day_features,
    on=["household_key", "CAMPAIGN"],
    how="left",
    validate="one_to_one"
)

modeling_table.loc[
    (modeling_table["household_key"] == 2337)
    & (modeling_table["CAMPAIGN"] == 13),
    ["household_key", "CAMPAIGN", "baskets_90d", "active_days_90d"]
]

,household_key,CAMPAIGN,baskets_90d,active_days_90d
4189,2337,13,257,81


In [24]:
invalid_active_days = (
    (modeling_table["active_days_90d"] < 0)
    | (modeling_table["active_days_90d"] > lookback_days)
    | (modeling_table["active_days_90d"] > modeling_table["baskets_90d"])
)

print(f"Modeling rows: {len(modeling_table):,}")
print(f"Missing active-day counts: {modeling_table['active_days_90d'].isna().sum():,}")
print(f"Invalid active-day counts: {invalid_active_days.sum():,}")

Modeling rows: 7,091
Missing active-day counts: 0
Invalid active-day counts: 0


In [25]:
numeric_feature_columns = [
    "baskets_90d",
    "retailer_sales_90d",
    "recency_days_90d",
    "has_transactions_90d",
    "avg_retailer_sales_per_basket_90d",
    "active_days_90d"
]

modeling_table_path = (
    project_root / "data" / "processed"
    / "household_campaign_modeling.parquet"
)

modeling_table.to_parquet(modeling_table_path, index=False)

print(f"Saved modeling examples: {len(modeling_table):,}")
print(f"Numerical features: {len(numeric_feature_columns)}")
print(f"File: {modeling_table_path}")

Saved modeling examples: 7,091
Numerical features: 6
File: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/processed/household_campaign_modeling.parquet


In [26]:
con.close()
print("Warehouse connection closed. Modeling table is saved.")

Warehouse connection closed. Modeling table is saved.
